# Evaluating the Brain Predictivity of Sentence Embeddings: A Comparison Between Large Language Models and Text Embedders

---

This notebook provides a code base useful to get started with project C. The main contents it covers are:
1. Loading fMRI responses from the [Pereira](https://www.nature.com/articles/s41467-018-03068-4) dataset;
2. Practical tips on how to fit the encoding models.

## Loading fMRI Responses

fMRI responses from the Pereira dataset can be accessed from two sources:
1. A GG Drive [folder](https://drive.google.com/drive/folders/1MjFOyJ9zYwec2bJqL4eNXO4Mlilt0oP8);
2. [This website](https://web.mit.edu/evlab/sites/default/files/documents/index2.html) (may be more convenient when downloading data from a terminal).


When running this notebook, it may be more convenient to access the data from GG Drive. You can do so by choosing the folder corresponding to one participant—here, we'll consider participant `M2`—and adding a shortcut to your GG Drive.

When working on Snellius, you can download data by completing the following steps:
1. Go to [this](https://web.mit.edu/evlab/sites/default/files/documents/index2.html) webpage;
2. Place your cursor on the link for one participant (e.g., M2), right-click, and copy the link;
3. Paste the link within the following command `! wget -O M02.tar "PASTE YOUR LINK HERE"`;
4. Run it in the command line (the exact command for participant M2, for example, is `! wget -O M02.tar "https://www.dropbox.com/s/n5yfb2cupd9zmwk/M02.tar?dl=1"`).

⚠️ Note: We are interested in data from experiments 2 and 3, but not all participants completed all experiments. This means you don't need to doenload the data for all participants, but only for those who completed experiments 2 and 3 (the [website](https://web.mit.edu/evlab/sites/default/files/documents/index2.html) indicates which participant completed what).

Whether you downloaded the compressed data in one way or the other, you'll need to unzip it by running the following command:

In [ ]:
# replace with the path from your GG Drive file system :)

!tar -xf 'ADD YOUR PATH/M02.tar'

Great, now we have the data for participant `M2`. We can go ahead and import a couple of useful libraries.

In [ ]:
import numpy as np
from pprint import pprint
from scipy.io import loadmat

In [ ]:
! ls /content/M02

data_180concepts_pictures.mat	 examples_180concepts_pictures.mat
data_180concepts_sentences.mat	 examples_180concepts_sentences.mat
data_180concepts_wordclouds.mat  examples_180concepts_wordclouds.mat
data_243sentences.mat		 examples_243sentences.mat
data_384sentences.mat		 examples_384sentences.mat


As you can see, the data we've downloaded includes many different files. For the purposes of the project, we'll only be interested in the following:

* `data_243sentences.mat`
* `data_384sentences.mat`

Let's try to open the first Matlab file and look at what's in there.

In [ ]:
data = loadmat('M02/data_243sentences.mat', simplify_cells=True)
data.keys()

/usr/local/lib/python3.12/dist-packages/scipy/io/matlab/_mio.py:235: MatReadWarning: Duplicate variable name "None" in stream - replacing previous with new
Considerscipy.io.matlab.varmats_from_mat to split file into single variable files
  matfile_dict = MR.get_variables(variable_names)


dict_keys(['__header__', '__version__', '__globals__', 'labels_task', 'labelsPassages', 'keyPassages', 'None', 'labelsPassageForEachSentence', 'keyPassageCategory', 'labelsPassageCategory', 'labelsSentences', 'keySentences', 'meta', 'examples_task', 'tstats_task', 'examples_passages', 'examples_passagesentences', '__function_workspace__'])

As mentioned in the project description (PDF file), you have the option to analyse brain responses either at the sentence or passage level, with the passage-level responses being an average of the responses to the 3-4 sentences making up the passage.

The responses at the passage-level are stored in `data['examples_passages']`

In [ ]:
data['examples_passages'].shape

(72, 170712)

72 is indeed the number of passages shown to participants in Experiment 3.

The sentence-level responses are stored in `data['examples_passagesentences']`

In [ ]:
data['examples_passagesentences'].shape

(243, 170712)

Again, 243 is the number of sentences from Experiment 3.

170712 is the number of voxels for which we have brain responses, but we're not interested in all of them—we should isolate the ones corresponding to the left-hemisphere (LH) language network. We can do so by taking advantage of the information stored in `data['meta']`, which will allow us to perform the following operations:

1. Map the brain network we're interested in to a number of specific sub-regions of interest (sub-ROIs)
2. Map the sub-ROIs to specific column indexes that we can use to slice the `data['examples_passagesentences']` or the `data['examples_passages']` arrays.

Let's start from 1. fMRI data was recorded in multiple functionally-localised brain networks, which we can see listed below.

In [ ]:
print(data['meta']['atlases'])
data['meta']['atlases'].shape

['aal_1ofN' 'languageParcelsConservative_aal' 'languageParcels_aal'
 'semantic_aal' 'multipleDemand_aal' 'MD' 'DMN' 'languageLH' 'languageRH'
 'visual_body' 'visual_face' 'visual_object' 'visual_scene' 'visual'
 'gordon']


(15,)

Each network corresponds to fMRI recordings from multiple sub-ROIs (listed in `data['meta']['rois']`). If we want to know which sub-ROIs correspond to `languageLH`, which is the 8th network in the list, we can look at the 8th array of sub-ROIs.

In [ ]:
target_index = np.where(data['meta']['atlases']=='languageLH')[0].item()
data['meta']['rois'][target_index]

array(['L_PostTemp', 'L_AntTemp', 'L_AngG', 'L_IFG', 'L_MFG', 'L_IFGorb'],
      dtype=object)

We can use the same index to identify the arrays of columns corresponding to the sub-ROIs in `languageLH`.

In [ ]:
data['meta']['roiColumns'][target_index].shape

(6,)

Great, now we can put all these arrays together into a single array of column indexes. Note that we have to subtract 1 to the original indexes to account for a mismatch between Python's and Matlab's indexing conventions.

In [ ]:
column_indexes = np.concat([arr-1 for arr in data['meta']['roiColumns'][target_index]], axis=0)
column_indexes.shape

(4930,)

Let's now use these indexes to slice our original array of fMRI responses.

In [ ]:
data['examples_passagesentences'][:, column_indexes].shape

(243, 4930)

As a last step, let's see how to access the original sentences/passages, so that we know what stimulus elicited each brain response.

In [ ]:
print(data['keySentences'][:3])

['Beekeeping encourages the conservation of local habitats.'
 "It is in every beekeeper's interest to conserve local plants that produce pollen."
 'As a passive form of agriculture, it does not require that native vegetation be cleared to make way for crops.']


`data['keySentences']` contains an ordered list of all the sentences presented to participants. By combining this information with that stored in `data['labelsPassageForEachSentence']`, it is also possible to understand which sentences belong to each passage.

In [ ]:
# see how this array looks like
print(data['labelsPassageForEachSentence'][:10])

# print sentences from passage 5
pprint(f"Passage 5:\n{' '.join(data['keySentences'][data['labelsPassageForEachSentence']==5])}")


[1 1 1 1 2 2 2 2 3 3]
('Passage 5:\n'
 'Each morning, participants in the study had to write down their dream '
 'experience from the previous night. They recorded if they recalled any '
 'dreams, and described each dream and its emotional intensity. Participants '
 'then assigned each dream to a category, such as a flying dream, a bad dream, '
 'or a nightmare.')


It is also possible to map passages to topics. The list of topics is provided in:

In [ ]:
data['keyPassageCategory']

array(['astronaut', 'beekeeping', 'blindness', 'bone_fracture', 'castle',
       'computer_graphics', 'dreams', 'gambling', 'hurricane',
       'ice_cream', 'infection', 'law_school', 'lawn_mower', 'opera',
       'owl', 'painter', 'pharmacist', 'polar_bear', 'pyramid',
       'rock_climbing', 'skiing', 'stress', 'taste', 'tuxedo'],
      dtype=object)

The following array allows determining which passages correspond to which topic. ⚠️ Note the Matlab indexing starting from 1.

In [ ]:
data['labelsPassageCategory']

array([ 2,  2,  2,  7,  7,  7,  8,  8,  8,  9,  9,  9, 10, 10, 10, 13, 13,
       13,  1,  1,  1,  6,  6,  6, 12, 12, 12, 17, 17, 17, 22, 22, 22, 24,
       24, 24,  3,  3,  3, 23, 23, 23,  4,  4,  4, 11, 11, 11, 14, 14, 14,
       16, 16, 16, 15, 15, 15, 18, 18, 18,  5,  5,  5, 19, 19, 19, 20, 20,
       20, 21, 21, 21], dtype=uint8)

Now you have all the elements to navigate the brain responses. Choose whether you want to focus on sentences or passages and adapt this code to your analyses :)

## Fitting Voxel-wise Encoding Models

Voxel-wise encoding models allow assessing the brain predictivity of model embeddings. Since we don't have a shared set of model embeddings, let's replace them with random vectors for the sake of this tutorial. Let's also assume that we're interested in modelling brain responses to _passages_.

In [ ]:
np.random.seed(5)
brain_responses = data['examples_passages'][:, column_indexes]
model_embeddings = np.random.rand(72, 512)
print(f"Dimensionality of brain responses: {brain_responses.shape}")
print(f"Dimensionality of fake model embeddings: {model_embeddings.shape}")

Dimensionality of brain responses: (72, 4930)
Dimensionality of fake model embeddings: (72, 512)


Let's install some useful libraries, including some standard ones and the `himalaya` library. This provides a Ridge regression implementation which, unlike the Scikit-learn one, selects a different regularisation parameter for each predicted variable, as we need when predicting brain responses.   

In [ ]:
%%capture
! pip install himalaya

In [ ]:
from sklearn.pipeline import make_pipeline
from himalaya.kernel_ridge import KernelRidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from scipy.stats import pearsonr

In [ ]:
n_splits = 5
kf = KFold(n_splits=n_splits)

# defining range of alpha values for the CV search
alphas = np.logspace(1, 20, 20)

# creating empty arrays to store results
_, n_voxels = brain_responses.shape
accs_train = np.empty((n_splits, n_voxels))
accs_test = np.empty((n_splits, n_voxels))

for i, (train_index, test_index) in enumerate(kf.split(model_embeddings)):

  # creating train and test splits
  X_train, X_test = model_embeddings[train_index, :], model_embeddings[test_index, :]
  Y_train, Y_test = brain_responses[train_index, :], brain_responses[test_index, :]

  # defining our modelling pipeline
  pipeline = make_pipeline(StandardScaler(with_mean=True, with_std=False),
                           KernelRidgeCV(alphas=alphas, cv=KFold(n_splits=5)))
  # fitting pipeline
  pipeline.fit(X_train, Y_train)

  # tracking accuracy on training and test set
  preds_train = pipeline.predict(X_train)
  corrs_train, _ = pearsonr(preds_train, Y_train, axis=0)
  accs_train[i] = corrs_train

  preds_test = pipeline.predict(X_test)
  corrs_test, _ = pearsonr(preds_test, Y_test, axis=0)
  accs_test[i] = corrs_test

Let's have a look at our accuracy values, averaged first across folds and then voxels.

In [ ]:
print(f"Training accuracy: {accs_train.mean(axis=0).mean():.2f}")
print(f"Test accuracy: {accs_test.mean(axis=0).mean():.2f}")

Training accuracy: 0.95
Test accuracy: -0.04


As you can see, even random vectors could achieve a high correlation with the true training fMRI responses. However, things change dramatically in the test set, where we see a near-zero correlation. By using actually meaningiful embeddings, you'll hopefully see much higher values.

Remember that this code assumes we're considering _one_ participant and representations from only _one_ model layer. In your experiments, this will have to be extended.

⚠️ The himalaya library is advertised as providing a CUDA-compatible implementation of Ridge regression. However, some model classes move back all tensors to CPU before fitting the regressions (even if you set `backend = set_backend("torch_cuda", on_error="warn")`). Please keep this in mind when submitting jobs on Snellius, and make sure you either optimise your code for GPUs or select a CPU partition.